# WeatherGPT - Person 4: Location Search and Map

**Repo:** https://github.com/sebin-droid/weather_gpt

---

## Files You Own

| File | What it does |
|------|--------------|
| `services/location_service.py` | Converts city name to lat/lon (geocoding) |
| `routers/location.py` | Web endpoint `/location/search?q=delhi` |
| `frontend/js/map.js` | Interactive map + moves pin on city search |

## Files You Must NOT Touch
- `main.py` (Person 1)
- `frontend/index.html` and `frontend/js/chat.js` (Person 2)
- `services/weather_service.py` (Person 3)

> **Run each cell top to bottom. Read the explanation before each code cell.**


---
# PART 0 - Common Setup

## Key Concepts

**Git** = Tracks every change to your code. Like a save history.

**GitHub** = Website storing the shared project online.

**Branch** = Your personal copy of the code. Your changes stay separate until merged.

**Virtual Environment (venv)** = Private box for your Python packages.

**FastAPI** = Python library that creates web endpoints (URLs that return data).

**Geocoding** = Converting a city name (Delhi) into GPS numbers (lat=28.65, lon=77.23).

**Latitude** = How far North/South. India is roughly 8N to 37N.

**Longitude** = How far East/West. India is roughly 68E to 97E.

**Leaflet.js** = Free JavaScript library that draws interactive maps using OpenStreetMap tiles.


## Step 0.1 - Verify Git and Python are Installed


In [ ]:
import subprocess, sys

git = subprocess.run(['git', '--version'], capture_output=True, text=True)
print('Git:', git.stdout.strip() or 'NOT FOUND - install from https://git-scm.com')
print('Python:', sys.version)
print()
print('You should see version numbers. If yes, you are ready!')


## Step 0.2 - Clone the Repository

`git clone` downloads the project from GitHub. Do this ONLY ONCE.

**Before running:** Sebin must add you as a GitHub Collaborator:
GitHub repo -> Settings -> Collaborators -> Add people -> your username

**Change PARENT_FOLDER** below to where you want the project saved.


In [ ]:
import subprocess, os

# CHANGE THIS to where you want the project
PARENT_FOLDER = r'C:\\Users\\ASUS\\Desktop\\sem3'
REPO_URL      = 'https://github.com/sebin-droid/weather_gpt.git'
REPO_FOLDER   = os.path.join(PARENT_FOLDER, 'weather_gpt')

if os.path.exists(REPO_FOLDER):
    print('Already cloned at:', REPO_FOLDER)
else:
    print('Cloning from', REPO_URL, '...')
    result = subprocess.run(
        ['git', 'clone', REPO_URL],
        capture_output=True, text=True, cwd=PARENT_FOLDER
    )
    if result.returncode == 0:
        print('Clone successful! Project at:', REPO_FOLDER)
    else:
        print('Clone failed:', result.stderr)
        print('Make sure Sebin added you as Collaborator and you accepted the invite.')


## Step 0.3 - Create Your Feature Branch

A branch is your personal workspace.
`git checkout -b feature/yourname` creates and switches to it in one step.

**Change 'yourname'** to your actual first name (e.g. feature/arjun)


In [ ]:
import subprocess

# CHANGE THIS to your first name
BRANCH_NAME = 'feature/yourname'
REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'

current = subprocess.run(
    ['git', 'branch', '--show-current'],
    capture_output=True, text=True, cwd=REPO_FOLDER
).stdout.strip()
print('Current branch:', current)

if current == BRANCH_NAME:
    print('Already on branch:', BRANCH_NAME)
else:
    r = subprocess.run(['git', 'checkout', '-b', BRANCH_NAME],
                       capture_output=True, text=True, cwd=REPO_FOLDER)
    if r.returncode == 0:
        print('Created and switched to:', BRANCH_NAME)
    else:
        r2 = subprocess.run(['git', 'checkout', BRANCH_NAME],
                            capture_output=True, text=True, cwd=REPO_FOLDER)
        msg = BRANCH_NAME if r2.returncode == 0 else 'ERROR: ' + r.stderr
        print('Switched to:', msg)

branches = subprocess.run(['git', 'branch'], capture_output=True, text=True, cwd=REPO_FOLDER)
print('All branches (* = current):'); print(branches.stdout)


## Step 0.4 - Set Up Python Virtual Environment

A virtual environment is a private box for your project's Python packages.
This cell creates venv/ and installs fastapi, uvicorn, requests.


In [ ]:
import subprocess, os, sys

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
VENV_PATH   = os.path.join(REPO_FOLDER, 'venv')
PIP_EXE     = os.path.join(VENV_PATH, 'Scripts', 'pip.exe')

if os.path.exists(VENV_PATH):
    print('Virtual environment already exists')
else:
    print('Creating virtual environment ...')
    subprocess.run([sys.executable, '-m', 'venv', VENV_PATH], check=True)
    print('Created!')

PACKAGES = ['fastapi', 'uvicorn', 'requests']
print('Installing:', PACKAGES, '...')
result = subprocess.run([PIP_EXE, 'install'] + PACKAGES, capture_output=True, text=True)
if result.returncode == 0:
    print('All packages installed!')
    for pkg in PACKAGES:
        v = subprocess.run([PIP_EXE, 'show', pkg], capture_output=True, text=True)
        for line in v.stdout.splitlines():
            if line.startswith('Version'):
                print(f'  {pkg}: {line}')
else:
    print('Error:', result.stderr[-300:])


## Step 0.5 - Test the Existing Backend

Before touching any code, verify the backend loads without errors.

To run the backend server manually, open a terminal and run:
```
cd C:\\Users\\ASUS\\Desktop\\sem3\\weather_gpt
.\\venv\\Scripts\\uvicorn.exe main:app --reload
```
Then open http://127.0.0.1:8000
Expected: `{"message": "WeatherGPT Backend is Running!"}`


In [ ]:
import subprocess, os

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
PYTHON_EXE  = os.path.join(REPO_FOLDER, 'venv', 'Scripts', 'python.exe')

test = subprocess.run(
    [PYTHON_EXE, '-c',
     'import sys; sys.path.insert(0, "."); from main import app; print("App loaded:", app.title)'],
    capture_output=True, text=True, cwd=REPO_FOLDER
)
if test.returncode == 0:
    print('Backend imports successfully!')
    print(' ', test.stdout.strip())
else:
    print('Error:', test.stderr[-300:])


## Step 0.6 - Git Golden Rules

| Rule | Why |
|------|-----|
| Only edit YOUR files | Prevents breaking teammates work |
| Commit every 20-30 minutes | Acts as a save point |
| Push to your own branch | `git push origin feature/yourname` |
| Never push directly to main | Only Person 1 does this at merge time |

The 3 Git commands you will use most:
```bash
git add .                              # Stage ALL changed files
git commit -m "what I just did"        # Save a snapshot
git push origin feature/yourname       # Upload to GitHub
```


In [ ]:
import subprocess

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'

branch = subprocess.run(['git', 'branch', '--show-current'],
    capture_output=True, text=True, cwd=REPO_FOLDER)
print('Current branch:', branch.stdout.strip())

status = subprocess.run(['git', 'status', '--short'],
    capture_output=True, text=True, cwd=REPO_FOLDER)
print('Changed files:', status.stdout.strip() or '(nothing changed)')


---
# PART 4 - Location Search and Map

## How Your 3 Files Connect to the Whole App

```
User types: "What is the weather in Mumbai?"
                   |
                   v
         chat.js sends to /chat endpoint
                   |
                   v
  Person 1 calls get_location('Mumbai')  <- YOUR location_service.py
                   |
                   v
  Returns: lat=19.07, lon=72.88, city=Mumbai, country=India
                   |
                   v
  weather_service.py fetches weather for those coordinates
                   |
                   v
  Answer shown in chat + chat.js calls window.updateMap()
                   |
                   v
  YOUR map.js moves map to Mumbai, drops a pin
```

**Without your code, no city can be found. The whole app breaks.**
Your code is the GPS of WeatherGPT.


---
## FILE 1 of 3: services/location_service.py

**What it does:** Takes a city name -> asks Open-Meteo Geocoding API -> returns lat, lon, city, country.

**API:** `https://geocoding-api.open-meteo.com/v1/search`
Free. No account. No API key.

### Code Walkthrough (read this before the cell below)

```python
city = city.strip()         # Remove spaces: '  Kochi  ' -> 'Kochi'
if not city: return None    # Guard: blank input -> stop immediately

params = {
    'name': city,           # City to search for
    'count': 1,             # Return only top 1 result
    'language': 'en'        # Names in English
}

response = requests.get(url, params=params)  # Call the API
data = response.json()                       # Convert response to Python dict
results = data.get('results')               # Get list of matching cities
place = results[0]                          # Take the best match

return {
    'city':      place['name'],
    'country':   place.get('country', ''),
    'latitude':  place['latitude'],
    'longitude': place['longitude']
}
```


In [ ]:
import os

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
FILE_PATH   = os.path.join(REPO_FOLDER, 'services', 'location_service.py')

CODE = 'import requests\n\n\ndef get_location(city: str):\n    # Convert a city name to geographic coordinates (latitude + longitude).\n    # Geocoding: turning a human-readable place name into GPS numbers.\n    # Uses the Open-Meteo Geocoding API -- FREE, no API key needed.\n    # Parameters: city (str) e.g. "Delhi" or "Thiruvananthapuram"\n    # Returns: dict with city, country, latitude, longitude\n    #          None if city not found or input is blank\n\n    # .strip() removes extra spaces: "  Kochi  " becomes "Kochi"\n    city = city.strip()\n\n    # Guard: if city name is empty, return None immediately.\n    # This prevents sending a useless API request.\n    if not city:\n        return None\n\n    # Open-Meteo Geocoding API URL\n    url = "https://geocoding-api.open-meteo.com/v1/search"\n\n    # params = inputs sent to the API\n    # Appear in URL as: ?name=Delhi&count=1&language=en\n    params = {\n        "name": city,        # city to search\n        "count": 1,          # return only top 1 result\n        "language": "en"     # names in English\n    }\n\n    # Send the HTTP GET request to the API\n    response = requests.get(url, params=params)\n\n    # raise_for_status() throws an error if API returned an error code\n    response.raise_for_status()\n\n    # .json() converts response text into a Python dictionary\n    data = response.json()\n\n    # API returns "results" with a list of matching cities\n    results = data.get("results")\n    if not results:\n        return None  # City not found\n\n    # Take the first (best) result\n    place = results[0]\n\n    # Return only what the rest of the app needs\n    return {\n        "city":      place["name"],               # e.g. "Delhi"\n        "country":   place.get("country", ""),    # e.g. "India"\n        "latitude":  place["latitude"],           # e.g. 28.65195\n        "longitude": place["longitude"]           # e.g. 77.23149\n    }\n'

with open(FILE_PATH, 'w', encoding='utf-8') as f:
    f.write(CODE)

print('Written:', FILE_PATH)
print('Size:', os.path.getsize(FILE_PATH), 'bytes')


### Test FILE 1 - Does get_location() work with real API calls?

Expected:
- Real Indian city -> dict with country='India'
- Empty string -> None
- Fake city -> None


In [ ]:
import sys, os

sys.path.insert(0, r'C:\Users\ASUS\Desktop\sem3\weather_gpt')
from services.location_service import get_location

r = get_location('Delhi')
print('Test 1 - Delhi:', r)
assert r and r['country'] == 'India', 'FAIL: Delhi should return India'
print('  PASS')

r2 = get_location('Thiruvananthapuram')
print('Test 2 - Thiruvananthapuram:', r2)
assert r2, 'FAIL'
print('  PASS')

r3 = get_location('')
print('Test 3 - Empty string:', r3)
assert r3 is None, 'FAIL: empty string should return None'
print('  PASS')

r4 = get_location('xyznotacityatall999')
print('Test 4 - Fake city:', r4)
assert r4 is None, 'FAIL: fake city should return None'
print('  PASS')

print('Multiple cities:')
for city in ['Mumbai', 'Bengaluru', 'Chennai', 'Kolkata', 'Hyderabad']:
    r = get_location(city)
    ok = r and r['country'] == 'India'
    lat = round(r['latitude'], 2) if r else 'N/A'
    lon = round(r['longitude'], 2) if r else 'N/A'
    print(f'  {"OK" if ok else "FAIL"} {city}: lat={lat}, lon={lon}')

print('All tests passed!')


---
## FILE 2 of 3: routers/location.py

**What it does:** Creates `GET /location/search?q=<city>` that returns JSON coordinates.

**What is ?q=kochi in the URL?**
The `?` starts the query string. `q=kochi` passes 'kochi' as parameter `q`.
FastAPI reads `q` from the URL automatically.

**HTTP 404** = Not Found. Standard web error code.

### Code Walkthrough

```python
router = APIRouter()               # Mini-app for location routes

@router.get('/location/search')    # Register this URL
def location_search(q: str):       # q is read from URL automatically
    location = get_location(q)     # Call the service
    if location is None:
        raise HTTPException(404)   # City not found -> 404 error
    return location                # Found -> return as JSON
```


In [ ]:
import os

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
ROUTERS_DIR = os.path.join(REPO_FOLDER, 'routers')
INIT_PATH   = os.path.join(ROUTERS_DIR, '__init__.py')
ROUTER_PATH = os.path.join(ROUTERS_DIR, 'location.py')

# Create routers/ directory
os.makedirs(ROUTERS_DIR, exist_ok=True)
print('routers/ folder:', ROUTERS_DIR)

# Create __init__.py so Python can import from routers/
if not os.path.exists(INIT_PATH):
    open(INIT_PATH, 'w').close()
    print('Created routers/__init__.py')
else:
    print('routers/__init__.py already exists')

CODE = 'from fastapi import APIRouter, HTTPException\n\nfrom services.location_service import get_location\n\n# APIRouter groups related endpoints together.\n# Person 1 will connect this in main.py using:\n#   from routers.location import router as location_router\n#   app.include_router(location_router)\nrouter = APIRouter()\n\n\n@router.get("/location/search")\ndef location_search(q: str):\n    # Search for a city by name and return its coordinates.\n    # Endpoint: GET /location/search?q=<city_name>\n    # Example: GET http://127.0.0.1:8000/location/search?q=delhi\n    # Returns city, country, latitude, longitude as JSON\n    # Returns HTTP 404 if city not found\n    # q (str): City name -- FastAPI reads from URL query string automatically\n\n    # Call the location service to geocode the city name\n    location = get_location(q)\n\n    # If city not found, raise HTTP 404 (Not Found)\n    # FastAPI formats this as JSON error automatically\n    if location is None:\n        raise HTTPException(\n            status_code=404,\n            detail="Location not found"\n        )\n\n    # City found -- FastAPI converts this dict to JSON automatically\n    return location\n'

with open(ROUTER_PATH, 'w', encoding='utf-8') as f:
    f.write(CODE)

print('Written:', ROUTER_PATH)
print('Size:', os.path.getsize(ROUTER_PATH), 'bytes')


### Test FILE 2 - Does the router import and register correctly?


In [ ]:
import sys
sys.path.insert(0, r'C:\Users\ASUS\Desktop\sem3\weather_gpt')

from routers.location import router

print('Router imported!')
print('Registered routes:')
for route in router.routes:
    print(' ', list(route.methods), route.path)

paths = [r.path for r in router.routes]
assert '/location/search' in paths, 'FAIL: /location/search not registered!'
print('/location/search correctly registered!')


### What to Tell Person 1 (Backend Lead who owns main.py)

After your code is done, tell Person 1 to add these 2 lines to `main.py`.
This connects your router to the main app.


In [ ]:
print('=' * 60)
print('MESSAGE FOR PERSON 1')
print('=' * 60)
print('Hi! I finished my location router.')
print('Add these 2 lines to main.py at the next merge window:')
print()
print('  # Near the top imports:')
print('  from routers.location import router as location_router')
print()
print('  # Right after app = FastAPI(...):')
print('  app.include_router(location_router)')
print()
print('This adds the /location/search endpoint to the app.')
print('=' * 60)


### Live API Test - Call /location/search with Real HTTP Requests

This cell starts the backend server and tests the endpoint.
You can also open your browser to: `http://127.0.0.1:8001/location/search?q=Chennai`


In [ ]:
import subprocess, time, os, json, urllib.request, urllib.error, re

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
UVICORN_EXE = os.path.join(REPO_FOLDER, 'venv', 'Scripts', 'uvicorn.exe')

main_py = open(os.path.join(REPO_FOLDER, 'main.py')).read()
if 'location_router' not in main_py:
    patched = main_py.replace(
        'app = FastAPI(',
        'from routers.location import router as location_router\napp = FastAPI('
    )
    patched = re.sub(r'(version="[^"]+"\s*\))', r'\1\napp.include_router(location_router)', patched)
    temp = os.path.join(REPO_FOLDER, '_testmain.py')
    with open(temp, 'w') as f: f.write(patched)
    main_mod = '_testmain'
    print('Note: Using patched main.py for this test')
else:
    main_mod = 'main'
    print('main.py already has location_router!')

server = subprocess.Popen(
    [UVICORN_EXE, f'{main_mod}:app', '--host', '127.0.0.1', '--port', '8001'],
    cwd=REPO_FOLDER, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print('Starting server ...'); time.sleep(3)

print('Testing /location/search:')
for city in ['Delhi', 'Mumbai', 'Chennai', 'Thiruvananthapuram']:
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:8001/location/search?q={city}', timeout=5) as r:
            d = json.loads(r.read())
        print(f'OK {city}: lat={d["latitude"]}, lon={d["longitude"]}, country={d["country"]}')
    except Exception as e:
        print(f'FAIL {city}: {e}')

try:
    urllib.request.urlopen('http://127.0.0.1:8001/location/search?q=xyzfake', timeout=5)
    print('FAIL: Should have been 404')
except urllib.error.HTTPError as e:
    print(f'OK Fake city -> HTTP {e.code} Not Found (correct!)')

server.terminate()
if main_mod == '_testmain':
    os.remove(temp)
print('Endpoint tests complete!')


---
## FILE 3 of 3: frontend/js/map.js

**What it does:** Draws an interactive map in `<div id='map'>`. When a city is found, moves the map there and drops a pin.

**How it connects to chat.js (Person 2):**
Person 2's chat.js already has:
```javascript
if (data.location && window.updateMap) {
    window.updateMap(data.location);  // calls YOUR function in map.js
}
```

**Why `window.updateMap` instead of a regular function?**
A plain JS function is only visible inside map.js.
Putting it on `window` makes it globally accessible to ALL scripts on the page.

**What is OpenStreetMap?**
A free community-built world map. Leaflet downloads 256x256 pixel tile images from OSM servers.

### Code Walkthrough
```javascript
let map = L.map('map').setView([10.0, 76.3], 6); // Kerala, zoom=6
L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
    attribution: '(c) OpenStreetMap contributors'
}).addTo(map);
let marker = null;  // no pin yet

window.updateMap = function(location) {
    map.setView([location.latitude, location.longitude], 9); // move camera
    if (marker) map.removeLayer(marker);  // remove old pin
    marker = L.marker([lat, lon])         // add new pin
        .addTo(map)
        .bindPopup('city, country')       // popup text
        .openPopup();                     // show popup
};
```


In [ ]:
import os

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
JS_DIR      = os.path.join(REPO_FOLDER, 'frontend', 'js')
MAP_PATH    = os.path.join(JS_DIR, 'map.js')

os.makedirs(JS_DIR, exist_ok=True)

CODE = "// ============================================================\n// map.js -- Interactive Map for WeatherGPT\n// Person 4 owns this file.\n// ============================================================\n//\n// HOW THIS CONNECTS TO THE REST OF THE APP:\n//   chat.js (Person 2) already calls:\n//       window.updateMap(data.location)\n//   where data.location = { city, country, latitude, longitude }\n//\n// LIBRARY USED: Leaflet.js, loaded in index.html via CDN.\n// The global 'L' variable is already available here.\n// ============================================================\n\n\n// --- STEP 1: Create the map ---\n// L.map('map') creates the map inside <div id='map'>\n// .setView([lat, lon], zoom) sets the starting camera\n// Zoom 6 = state/region level (1=whole world, 18=street level)\nlet map = L.map('map').setView([10.0, 76.3], 6);\n\n\n// --- STEP 2: Add OpenStreetMap tiles ---\n// L.tileLayer() downloads 256x256 map images from OpenStreetMap servers.\n// URL variables: {s}=subdomain, {z}=zoom, {x}=column, {y}=row\n// attribution = copyright text (required by OpenStreetMap license)\nL.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {\n    attribution: '&copy; OpenStreetMap contributors'\n}).addTo(map);\n\n\n// --- STEP 3: Track the current pin ---\n// Starts as null (no pin yet).\n// When updateMap() is called, we remove the old pin and add a new one.\nlet marker = null;\n\n\n// --- STEP 4: The updateMap function ---\n// Attached to window so ALL scripts can call it.\n// chat.js calls: window.updateMap(data.location)\n// location = { city: 'Delhi', country: 'India', latitude: 28.65, longitude: 77.23 }\nwindow.updateMap = function(location) {\n\n    const lat = location.latitude;    // e.g. 28.65195\n    const lon = location.longitude;   // e.g. 77.23149\n\n    // Move map camera to the new city (zoom 9 = city level)\n    map.setView([lat, lon], 9);\n\n    // Remove old pin so they do not pile up\n    if (marker) {\n        map.removeLayer(marker);\n    }\n\n    // Add new pin with a popup showing city name\n    // .bindPopup() attaches a text bubble to the pin\n    // .openPopup() makes the bubble appear immediately\n    marker = L.marker([lat, lon])\n        .addTo(map)\n        .bindPopup(`${location.city}, ${location.country || ''}`)\n        .openPopup();\n};\n"

with open(MAP_PATH, 'w', encoding='utf-8') as f:
    f.write(CODE)

print('Written:', MAP_PATH)
print('Size:', os.path.getsize(MAP_PATH), 'bytes')


### Test FILE 3 - Verify map.js has all required elements


In [ ]:
import os

MAP_PATH = os.path.join(r'C:\Users\ASUS\Desktop\sem3\weather_gpt', 'frontend', 'js', 'map.js')
js = open(MAP_PATH, encoding='utf-8').read()

checks = [
    ("L.map('map')",         'Map created on the #map div'),
    ('setView([10.0, 76.3]',  'Default view on Kerala'),
    ('tile.openstreetmap',    'OpenStreetMap tiles loaded'),
    ('let marker = null',     'Marker initialized to null'),
    ('window.updateMap',      'updateMap exposed globally'),
    ('location.latitude',     'Latitude read from location'),
    ('location.longitude',    'Longitude read from location'),
    ('map.setView',           'Camera moves to new city'),
    ('map.removeLayer',       'Old marker removed'),
    ('L.marker',              'New pin created'),
    ('bindPopup',             'Popup attached to pin'),
    ('location.city',         'City name in popup'),
]

print('Verifying map.js:')
all_ok = True
for snippet, desc in checks:
    found = snippet in js
    print(f'{"OK" if found else "FAIL"} {desc}')
    if not found:
        print(f'  Missing: {snippet}')
        all_ok = False

print('map.js is perfect!' if all_ok else 'Some elements missing - re-run write cell.')


### Visual Test - Open the Map in Your Browser

Run this cell to start the frontend and see the map.
Click the city buttons to watch the map move and the pin drop.


In [ ]:
import subprocess, os, webbrowser, time

REPO_FOLDER  = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
FRONTEND_DIR = os.path.join(REPO_FOLDER, 'frontend')
INDEX_PATH   = os.path.join(FRONTEND_DIR, 'index.html')

os.makedirs(os.path.join(FRONTEND_DIR, 'css'), exist_ok=True)

HTML = '<!DOCTYPE html>\n<html lang=\'en\'>\n<head>\n  <meta charset=\'UTF-8\'>\n  <title>WeatherGPT - Map Test (Person 4)</title>\n  <link rel=\'stylesheet\' href=\'https://unpkg.com/leaflet@1.9.4/dist/leaflet.css\' />\n  <style>\n    body { font-family: Arial, sans-serif; background: #f4f7fb; margin: 0; }\n    .app { max-width: 600px; margin: 20px auto; padding: 20px; background: white;\n           border-radius: 10px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }\n    h1 { color: #1e40af; }\n    #map { height: 400px; border-radius: 8px; margin-top: 15px; }\n    button { padding: 10px 16px; background: #2563eb; color: white;\n             border: none; border-radius: 8px; cursor: pointer; margin: 4px; }\n    button:hover { background: #1d4ed8; }\n    p { color: #555; }\n  </style>\n</head>\n<body>\n  <div class=\'app\'>\n    <h1>WeatherGPT - Map Test (Person 4)</h1>\n    <p>Map starts centered on Kerala. Click a city to move the pin.</p>\n    <button onclick="testCity(28.65195,77.23149,\'Delhi\',\'India\')">Delhi</button>\n    <button onclick="testCity(19.07283,72.88261,\'Mumbai\',\'India\')">Mumbai</button>\n    <button onclick="testCity(13.08784,80.27847,\'Chennai\',\'India\')">Chennai</button>\n    <button onclick="testCity(12.97194,77.59369,\'Bengaluru\',\'India\')">Bengaluru</button>\n    <button onclick="testCity(8.4855,76.94924,\'Thiruvananthapuram\',\'India\')">Thiruvananthapuram</button>\n    <div id=\'map\'></div>\n  </div>\n  <script src=\'https://unpkg.com/leaflet@1.9.4/dist/leaflet.js\'></script>\n  <script src=\'js/map.js\'></script>\n  <script>\n    function testCity(lat, lon, city, country) {\n      window.updateMap({ latitude: lat, longitude: lon, city: city, country: country });\n    }\n  </script>\n</body>\n</html>\n'

with open(INDEX_PATH, 'w', encoding='utf-8') as f:
    f.write(HTML)

for name in ['voice.js', 'chat.js']:
    p = os.path.join(FRONTEND_DIR, 'js', name)
    if not os.path.exists(p):
        with open(p, 'w') as f: f.write(f'// {name} placeholder\n')

server = subprocess.Popen(
    ['python', '-m', 'http.server', '5500'],
    cwd=FRONTEND_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(2)
webbrowser.open('http://127.0.0.1:5500')

print('Frontend running at http://127.0.0.1:5500')
print('WHAT TO CHECK:')
print('1. Map shows Kerala/South India by default')
print('2. Click Delhi -> map moves to Delhi, pin drops')
print('3. Click Mumbai -> map moves to Mumbai, pin moves')
print('4. Popup shows city name on the pin')
print('All 4 working = map.js is correct!')
print('To stop: server.terminate()')


---
## Summary - Verify All Your Files


In [ ]:
import os

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'

files = [
    ('services/location_service.py', 'Geocoding - city name to lat/lon'),
    ('routers/__init__.py',          'Makes routers/ a Python package'),
    ('routers/location.py',          'Endpoint /location/search'),
    ('frontend/js/map.js',           'Interactive map + updateMap()'),
]

print('FILES OWNED BY PERSON 4:')
print('=' * 55)
all_good = True
for rel, desc in files:
    full = os.path.join(REPO_FOLDER, rel)
    ok   = os.path.exists(full) and os.path.getsize(full) > 0
    size = os.path.getsize(full) if os.path.exists(full) else 0
    print(f'{"OK" if ok else "MISSING"} {rel}')
    print(f'   {desc} | {size} bytes')
    print()
    if not ok: all_good = False

print('All files in place!' if all_good else 'Some missing - re-run write cells above.')


---
## Git - Commit and Push Your Work

**When?** Every 20-30 minutes. A commit is a save point.

```bash
git add .                              # Mark changed files as ready
git commit -m "describe what I did"   # Save snapshot with message
git push origin feature/yourname       # Upload to GitHub
```

- `git add .` = tells Git to track all changed files
- `git commit -m` = saves a snapshot with a description
- `git push` = uploads to GitHub so teammates see your work

**Change BRANCH_NAME** to your branch before running.


In [ ]:
import subprocess

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
BRANCH_NAME = 'feature/yourname'    # CHANGE to your branch name
COMMIT_MSG  = 'Add real city search and interactive map (Person 4)'

def git(args):
    r = subprocess.run(['git'] + args, capture_output=True, text=True, cwd=REPO_FOLDER)
    return r.returncode, (r.stdout + r.stderr).strip()

_, out = git(['status', '--short'])
print('Changed files:', out or '(nothing changed)')

git(['add', '.'])
print('git add . done')

code, out = git(['commit', '-m', COMMIT_MSG])
print('git commit:', out[:200])

if code == 0 or 'nothing to commit' in out:
    code2, out2 = git(['push', 'origin', BRANCH_NAME])
    print('git push:', out2[:300])
    if code2 == 0:
        print('Pushed to GitHub!')
        print('Branch: https://github.com/sebin-droid/weather_gpt/tree/' + BRANCH_NAME)
    else:
        print('Push failed. Run in terminal: git push origin', BRANCH_NAME)


### Quick Commit - Run This Every 20-30 Minutes


In [ ]:
import subprocess

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
BRANCH_NAME = 'feature/yourname'   # CHANGE to your branch
COMMIT_MSG  = 'Work in progress'   # CHANGE to describe what you did

for args, label in [
    (['add', '.'],                  'Staging...'),
    (['commit', '-m', COMMIT_MSG],  'Committing...'),
    (['push', 'origin', BRANCH_NAME], 'Pushing to GitHub...'),
]:
    print(label)
    r = subprocess.run(['git'] + args, capture_output=True, text=True, cwd=REPO_FOLDER)
    print(' ', (r.stdout + r.stderr).strip()[:200])
print('Done!')


### Pull Latest From Main - Run After Each Merge Window


In [ ]:
import subprocess

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
BRANCH_NAME = 'feature/yourname'  # CHANGE to your branch

def git(args):
    return subprocess.run(['git']+args, capture_output=True, text=True, cwd=REPO_FOLDER).stdout.strip()

print('Current branch:', git(['branch', '--show-current']))
print('Pulling from main...')
print(git(['pull', 'origin', 'main']))
print('Done! Restart the backend server to pick up new changes.')


---
## Final Checklist - Before Telling the Team You Are Done


In [ ]:
import os, sys, subprocess

REPO_FOLDER = r'C:\Users\ASUS\Desktop\sem3\weather_gpt'
sys.path.insert(0, REPO_FOLDER)

items = []

for f in ['services/location_service.py', 'routers/__init__.py',
          'routers/location.py', 'frontend/js/map.js']:
    p = os.path.join(REPO_FOLDER, f)
    ok = os.path.exists(p) and os.path.getsize(p) > 0
    items.append((f'File exists: {f}', ok))

try:
    from services.location_service import get_location
    items.append(('location_service.py imports', True))
    r = get_location('Delhi')
    items.append(("get_location('Delhi') -> India", bool(r and r.get('country') == 'India')))
    items.append(("get_location('') -> None", get_location('') is None))
except Exception as e:
    items.append((f'location_service error: {e}', False))

try:
    from routers.location import router
    paths = [rt.path for rt in router.routes]
    items.append(('/location/search route registered', '/location/search' in paths))
except Exception as e:
    items.append((f'router import error: {e}', False))

try:
    js = open(os.path.join(REPO_FOLDER, 'frontend', 'js', 'map.js')).read()
    has = all(x in js for x in ['window.updateMap','L.map','L.tileLayer','L.marker','bindPopup'])
    items.append(('map.js has all required code', has))
except Exception as e:
    items.append((f'map.js check: {e}', False))

log = subprocess.run(['git','log','--oneline','-3'],
    capture_output=True, text=True, cwd=REPO_FOLDER).stdout.strip()
items.append(('At least one git commit made', bool(log)))

print('PERSON 4 - FINAL CHECKLIST')
print('=' * 55)
passed = 0
for item, ok in items:
    print(f'{"OK" if ok else "FAIL"} {item}')
    if ok: passed += 1

print(f'Score: {passed}/{len(items)}')
print('All done! Tell Person 1 to add your 2 lines.' if passed == len(items)
      else 'Fix the failing items by re-running the write cells.')
print()
print('Recent commits:')
print(log or '(no commits yet)')
print()
print('MESSAGE FOR PERSON 1:')
print('  from routers.location import router as location_router')
print('  app.include_router(location_router)')


---
## How Your Work Connects to the Team

```
WeatherGPT App
|
+-- Person 1: Backend AI Chat
|     Uses YOUR get_location() to find cities from user questions
|
+-- Person 2: Frontend UI
|     Calls YOUR window.updateMap() to move the map pin
|
+-- Person 3: Weather and Alerts
|     Uses YOUR get_location() before fetching weather data
|
+-- Person 4 (YOU): Location and Map
|     services/location_service.py  -> geocodes any city name
|     routers/location.py           -> exposes /location/search endpoint
|     frontend/js/map.js            -> interactive map that follows the chat
|
+-- Person 5: Voice and Translation
```

### Demo flow (what judges will see):

1. User types: 'What is the weather in Mumbai?'
2. chat.js sends to /chat endpoint
3. Person 1 calls YOUR get_location('Mumbai') -> lat=19.07, lon=72.88
4. Weather fetched for those coordinates
5. Answer shown in chat
6. chat.js calls YOUR window.updateMap({city:'Mumbai', latitude:19.07,...})
7. YOUR map.js moves the map to Mumbai and drops a pin

Without your code:
- No city can be found
- The map stays blank forever
- The entire app breaks

Your code is the GPS of WeatherGPT!
